# Running LDPred2 on BMRC

In [1]:
setwd("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb")

In [2]:
library(bigsnpr)
library(data.table)
#options(bigstatsr.check.parallel.blas = FALSE)
#options(default.nproc.blas = NULL)

Loading required package: bigstatsr



In [3]:
path_phenotypes <- "data/phenotypes/curated_phenotypes.tsv"
path_sumstats <- "data/prs/sumstat/ukb_hapmap_500k_eur_Alanine_aminotransferase_residual_chr1.txt.gz"
path_bed <- "short_merged_ukb_hapmap_rand_10k_eur_chr1_3.bed"
path_ld_bed <- "data/prs/hapmap/ld/short_merged_ukb_hapmap_rand_10k_eur_chr1_3.bed"

In [4]:
file.exists(path_bed)
file.exists(path_ld_bed)

[1] FALSE

[1] TRUE

## Helper functions (some functions require active internet connection)

In [5]:
snp_asGeneticPosLocal <- function (infos.chr, infos.pos, mapdir, ncores = 1, 
    rsid = NULL, genetic_map = 'omni') 
{
    bigassertr::assert_package("R.utils")
    bigassertr::assert_lengths(infos.chr, infos.pos)
    if (!is.null(rsid)) 
        bigassertr::assert_lengths(rsid, infos.pos)
    snp_split(infos.chr, function(ind.chr, pos, dir, rsid) {
        chr <- attr(ind.chr, "chr")
        stopifnot(genetic_map %in% c('hapmap','omni'))
        map_origin <- ifelse(
            genetic_map %in% 'omni',
            ".OMNI.interpolated_genetic_map",
            ".interpolated_genetic_map")
        basename <- paste0("chr", chr, map_origin)
        gzfile <- file.path(mapdir, paste0(basename, '.gz'))
        mapfile <- file.path(mapdir, basename)
        if (!file.exists(mapfile)) {
            stopifnot(file.exists(gzfile))
            R.utils::gunzip(gzfile)
        }
        map.chr <- bigreadr::fread2(mapfile, showProgress = FALSE)
        if (is.null(rsid)) {
            ind <- bigutilsr::knn_parallel(as.matrix(map.chr$V2), 
                as.matrix(pos[ind.chr]), k = 1, ncores = 1)$nn.idx
            new_pos <- map.chr$V3[ind]
        }
        else {
            ind <- match(rsid[ind.chr], map.chr$V1)
            new_pos <- map.chr$V3[ind]
            indNA <- which(is.na(ind))
            if (length(indNA) > 0) {
                pos.chr <- pos[ind.chr]
                new_pos[indNA] <- suppressWarnings(stats::spline(pos.chr, 
                  new_pos, xout = pos.chr[indNA], method = "hyman")$y)
            }
        }
        new_pos
    }, combine = "c", pos = infos.pos, dir = dir, rsid = rsid, 
        ncores = ncores)
}


qc_sumstat <- function(info_snp) {
  chi2 <- with(info_snp, (beta / beta_se)^2)
  print(round(mean(chi2, na.rm = TRUE), 2))
  S <- rep(NA, ncol(G)); S[info_snp$`_NUM_ID_`] <- chi2
  signif <- pchisq(S, df = 1, lower.tail = FALSE) < 5e-8
  print(sum(signif, na.rm = TRUE))
  ind.keep <- snp_clumping(
    G, infos.chr = map$chr, infos.pos = map$pos, S = S,
    ind.row = ind.val, thr.r2 = 0.01, size = 10e3, ncores = NCORES,
    exclude = which(is.na(S) | !signif))
  print(length(ind.keep))
}

## 1. Load phenotypes

In [6]:
# Get phenotypes
phenotypes <- fread(path_phenotypes, nrows = 100)
colnames(phenotypes)[1] <- "IID"
phenotypes$FID <- phenotypes$IID

In [7]:
# get columns containing various metrics
#header <- colnames(phenotypes)
#cols_covs <- header %in% c('sex','age','ukbb.centre')
#cols_ids <- header %in% 'eid'
#cols_pcs <- grepl('PC[0-9]+', header)

In [8]:
# get tables for covs and pcs
#covs <- phenotypes[,cols_ids | cols_covs, with = FALSE]
#pcs <- phenotypes[,cols_ids | cols_pcs, with = FALSE]

## 2. Load and transform the summary statistic file

In [9]:
sumstats <- bigreadr::fread2(path_sumstats)

In [10]:
# make artificial rsid
sumstats$rsid <- 
    apply(
        data.frame(
            locus = sumstats$locus, 
            a0 = sumstats$a0,
            a1 = sumstats$a1
            ),
        1,
        paste, 
        collapse = '_')

In [11]:
# extract position and chromosome
locus <- do.call(rbind, strsplit(sumstats$locus, split = "\\:"))
colnames(locus) <- c('chr','pos')
sumstats <- cbind(sumstats, locus)

In [12]:
# calculate MAF
#sumstats$AF <- sumstats$AC / sumstats$AN
sumstats$MAF <- unlist(lapply(sumstats$AF, function(af) min(af, 1-af)))

In [13]:
sumstats <- 
    data.frame(
        chr = sumstats$chr,
        pos = as.integer(sumstats$pos),
        rsid = sumstats$rsid,
        a1 = sumstats$a1,
        a0 = sumstats$a0,
        n_eff = sumstats$n,
        beta_se = sumstats$standard_error,
        p = sumstats$p_value,
        beta = sumstats$beta,
        INFO = NA,
        MAF = sumstats$MAF
    )

## 2. QC summary stats

In [14]:
file.exists.ext <- function(fname, ext = ".txt"){
    fname <- tools::file_path_sans_ext(fname)
    fname <- paste0(fname, ext)
    return(file.exists(fname))
}

In [15]:
NCORES <- nb_cores()
tmp <- tempfile(tmpdir = "data/tmp/tmp-data")
on.exit(file.remove(paste0(tmp, ".sbk")), add = TRUE)

# preprocess the bed file (only need to do once for each data set) and attach 
if (!file.exists.ext(path_ld_bed, '.bk')) snp_readBed(path_ld_bed)
# # make check to see if .sbk already exists
basename <- tools::file_path_sans_ext(path_ld_bed)
rds <- paste0(basename,'.rds')
obj.bigSNP <- snp_attach(rds)

Warning message in file.remove(paste0(tmp, ".sbk")):
"cannot remove file 'data/tmp/tmp-data/filec6166c749f03.sbk', reason 'No such file or directory'"


In [16]:
# only sumstat table
table(sumstats$chr)


  chr1 
103901 

In [17]:
map <- obj.bigSNP$map[-3]
names(map) <- c("chr", "rsid", "pos", "a1", "a0")
info_snp <- bigsnpr::snp_match(sumstats, map, strand_flip = FALSE)

103,901 variants to be matched.

Some duplicates were removed.

97,611 variants have been matched; 0 were flipped and 0 were reversed.



In [18]:
table(obj.bigSNP$map$chr)


 chr1  chr2 
97826 99504 

In [19]:
G <- obj.bigSNP$genotypes

In [20]:
sum(is.na(as.integer(gsub('chr','',map$chr))))

[1] 0

In [21]:
chi2 <- with(info_snp, (beta / beta_se)^2)
print(round(mean(chi2, na.rm = TRUE), 2))
S <- rep(NA, ncol(G)); S[info_snp$`_NUM_ID_`] <- chi2
signif <- pchisq(S, df = 1, lower.tail = FALSE) < 5e-8
print(sum(signif, na.rm = TRUE))
ind.keep <- snp_clumping(
    G, infos.chr = as.integer(gsub('chr','',map$chr)), infos.pos = map$pos, S = S,
    thr.r2 = 0.01, size = 10e3, ncores = NCORES,
    exclude = which(is.na(S) | !signif)
)
print(length(ind.keep))

[1] 1.45
[1] 97
[1] 6


In [22]:
sd_val <- sd(info_snp$`_NUM_ID_`)
sd_ss <- with(info_snp, 2 / sqrt(n_eff * beta_se^2))

In [25]:
is_bad <-
  sd_ss < (0.5 * sd_val) | sd_ss > (sd_val + 0.1) | sd_ss < 0.1 | sd_val < 0.05
qplot(sd_val, sd_ss, color = is_bad, alpha = I(0.5)) +
  theme_bigstatsr() +
  coord_equal() +
  scale_color_viridis_d(direction = -1) +
  geom_abline(linetype = 2, color = "red") +
  labs(x = "Standard deviations in the validation set",
       y = "Standard deviations derived from the summary statistics",
       color = "Removed?")

sum(is_bad,na.rm = TRUE)
length(is_bad)


ERROR: Error in qplot(sd_val, sd_ss, color = is_bad, alpha = I(0.5)): could not find function "qplot"


## 3. Calculate LD Matrix

In [23]:
## copied from https://choishingwan.github.io/PRS-Tutorial/ldpred/
# Get maximum amount of cores
NCORES <- nb_cores()
tmp <- tempfile(tmpdir = "data/tmp/tmp-data")
on.exit(file.remove(paste0(tmp, ".sbk")), add = TRUE)

# Initialize variables for storing the LD score and LD matrix
corr <- NULL
ld <- NULL

# We want to know the ordering of samples in the bed file 
fam.order <- NULL

# preprocess the bed file (only need to do once for each data set) and attach 
#snp_readBed(path_ld_bed)
basename <- tools::file_path_sans_ext(path_ld_bed)
rds <- paste0(basename,'.rds')
obj.bigSNP <- snp_attach(rds)

Warning message in file.remove(paste0(tmp, ".sbk")):
"cannot remove file 'data/tmp/tmp-data/filec6165da3683d.sbk', reason 'No such file or directory'"


In [24]:
# extract the SNP information from the genotype
map <- obj.bigSNP$map[-3]
names(map) <- c("chr", "rsid", "pos", "a1", "a0")

# perform SNP matching
info_snp <- bigsnpr::snp_match(sumstats, map, strand_flip = FALSE)
#info_snp <- snp_match(sumstats, map)

# Assign the genotype to a variable for easier downstream analysis
genotype <- obj.bigSNP$genotypes

103,901 variants to be matched.

Some duplicates were removed.

97,611 variants have been matched; 0 were flipped and 0 were reversed.



In [25]:
table(info_snp$chr)


 chr1 
97611 

In [26]:
# Rename the data structures
CHR <- as.numeric(gsub('chr','',map$chr))
POS <- map$pos
table(map$chr[is.na(CHR)])
table(CHR)
#CHR <- CHR[!is.na(CHR)] # omit alternate contigs like 'chr8_KI270821v1_alt'

< table of extent 0 >

CHR
    1     2 
97826 99504 

In [29]:



# get the CM information from 1000 Genome
# will download the 1000G file to the current directory (".")
#snp_asGeneticPos(CHR, POS, dir = ".")
POS2 <- snp_asGeneticPosLocal(CHR, POS, mapdir = "data/prs/1000-genomes-genetic-maps",genetic_map = 'hapmap')

In [30]:
matches <- sum(POS2==0)/length(POS2) # hapmap has many less missing variants than omni
msg <- paste0((1-round(matches,3))*100,'% of LD panel variants are in genetic map.')
print(msg)

[1] "100% of LD panel variants are in genetic map."


In [ ]:
chrs <- paste0("chr",1:22)
which(info_snp$chr == "chr2")

In [ ]:
table(info_snp$chr)

In [ ]:
# required args

In [ ]:
est_corr_matrix <- function(info_snp, genotype, chroms = 1:10, build = 'GRCh38'){
    
    if (build %in% 'GRCh38') chroms <- paste0("chr",chroms)
    stopifnot(all(unique(info_snp$chr) %in% chrs))
    for (chr in chrs) {
        # Extract SNPs that are included in the chromosome
        ind.chr <- which(info_snp$chr == chr)
        ind.chr2 <- info_snp$`_NUM_ID_`[ind.chr]
        stopifnot(length(ind.chr) > 0)
        # Calculate the LD
        corr0 <- snp_cor(
                genotype,
                ind.col = ind.chr2,
                ncores = NCORES,
                infos.pos = POS2[ind.chr2],
                size = 3 / 1000
            )
        if (chr == "chr1") {
            ld <- Matrix::colSums(corr0^2)
            corr <- as_SFBM(corr0, tmp)
        } else {
            ld <- c(ld, Matrix::colSums(corr0^2))
            corr$add_columns(corr0, nrow(corr))
        }
    }
    
    return(list(ld = ld, corr = corr))
}

In [31]:
# calculate LD
chrs <- paste0("chr",1)
stopifnot(all(unique(info_snp$chr) %in% chrs))
for (chr in chrs) {
    # Extract SNPs that are included in the chromosome
    ind.chr <- which(info_snp$chr == chr)
    ind.chr2 <- info_snp$`_NUM_ID_`[ind.chr]
    stopifnot(length(ind.chr) > 0)
    # Calculate the LD
    corr0 <- snp_cor(
            genotype,
            ind.col = ind.chr2,
            ncores = NCORES,
            infos.pos = POS2[ind.chr2],
            size = 3 / 1000
        )
    if (chr == "chr1") {
        ld <- Matrix::colSums(corr0^2)
        corr <- as_SFBM(corr0, tmp)
    } else {
        ld <- c(ld, Matrix::colSums(corr0^2))
        corr$add_columns(corr0, nrow(corr))
    }
}
print('done')
# We assume the fam order is the same across different chromosomes
fam.order <- as.data.table(obj.bigSNP$fam)
# Rename fam order
setnames(fam.order,
        c("family.ID", "sample.ID"),
        c("FID", "IID"))

[1] "done"


## 4. Perform LD score regression

In [46]:
df_beta <- info_snp[,c("beta", "beta_se", "n_eff", "_NUM_ID_")]
ldsc <- snp_ldsc(   ld, 
                    length(ld), 
                    chi2 = (df_beta$beta / df_beta$beta_se)^2,
                    sample_size = df_beta$n_eff, 
                    blocks = NULL)
h2_est <- ldsc[["h2"]]## 4. Perform LD score regression
ldsc_h2_est <- ldsc[["h2"]]

In [47]:
ldsc

int        h2 
1.0224673 0.0115691

## 4. Perform LD score regression

In [48]:
beta_inf <- snp_ldpred2_inf(corr, df_beta, h2 = h2_est)

In [49]:
length(beta_inf)

[1] 97611

In [50]:
set.seed(1)
ind.val <- sample(nrow(G), 350)
ind.test <- setdiff(rows_along(G), ind.val)

In [51]:
y   <- obj.bigSNP$fam$affection

In [55]:
beta_inf <- snp_ldpred2_inf(corr, df_beta, ldsc_h2_est)
pred_inf <- big_prodVec(G, beta_inf, ind.col = df_beta$`_NUM_ID_`)

In [56]:
length(pred_inf)

[1] 10000

In [1]:
hist(pred_inf)

ERROR: Error in hist(pred_inf): object 'pred_inf' not found
